In [51]:
import numpy as np
from matplotlib import pyplot as plt
import string
import random
import re
import requests
import os
import textwrap

In [74]:
# the language model

# initialize Markov matrix
M = np.ones((26,26))

# initial state distribution
pi = np.zeros(26)

# a function to update the markov matrix
def update_transition(ch1, ch2):
  i = ord(ch1) - 97
  j = ord(ch2) - 97
  M[i,j] += 1

# a function to update the initial state distribution
def update_pi(ch):
  i = ord(ch) - 97
  pi[i] += 1

# get the log probability of a word/token
def get_word_prob(word):
  i = ord(word[0]) - 97
  logp = np.log(pi[i])

  for ch in word[1:]:
    j = ord(ch) - 97
    logp += np.log(M[i,j])
    i = j


  return logp

# get the probability of a sequence of words
def get_sequence_probs(words):
  if type(words) == str:
    words= words.split()

  logp = 0
  for word in words:
    logp+= get_word_prob(word)
  return logp


In [75]:
# # Create Subsitution Ciphers

# one will acts as the key other as value

letters1 = list(string.ascii_lowercase)
letters2 = list(string.ascii_lowercase)

true_mapping = {}

# shuffle second set of letters
random.shuffle(letters2)

# populate map
for k, v in zip(letters1, letters2):
  true_mapping[k] = v


In [76]:
# download the file

if not os.path.exists('/content/melville-moby_dick.txt'):
  print('Downloading the file')
  r = requests.get('https:lazyprogrammer.me/course_files/moby_dick.txt')
  with open('moby_dick.txt','w') as f:
    f.write(r.content.decode())


In [84]:
def get_word_prob(word):
    global pi  # Ensure `pi` is accessible within the function

    if word and word[0].isalpha():  # Check if the word starts with an alphabet character
        char_index = ord(word[0].lower()) - ord('a')  # Calculate index for the first character
        if 0 <= char_index < len(pi):  # Check if the index is within bounds of `pi`
            log_prob = np.log(pi[char_index])  # Access probability from `pi` array
            for ch in word[1:]:
                # Additional processing for subsequent characters (if needed)
                pass
            return log_prob
    return float('-inf')  # Return a default value for invalid cases


In [85]:
# for replacing non-alpha characters
regex = re.compile('[^a-zA-Z]')

# load in words
for line in open('/content/melville-moby_dick.txt'):
  line = line.rstrip()

  # there are blank lines in the file
  if line:
    # replace all non-alpha characters with space
    line = regex.sub(' ', line)

    # split the tokens in the line and lowercase
    tokens = line.lower().split()

    for token in tokens:

      # first letter
      ch0 = token[0]
      update_pi(ch0)

      # other letters
      for ch1 in token[1:]:
        update_transition(ch0, ch1)
        ch0 = ch1


# Normalize the probabilities

pi = pi.sum()
M /= M.sum(axis =1, keepdims = True)




IndexError: invalid index to scalar variable.

In [ ]:
original_message = '''I then lounged down the street and found as I expected
'''

In [86]:
# function to encode a message
def encode_message(msg):
  msg = msg.lower()

  msg = regex.sub(' ', msg)

  coded_msg = []
  for ch in msg:
    coded_ch = ch
    if ch in true_mapping:
      coded_ch = true_mapping[ch]
    coded_msg.append(coded_ch)

  return ''.join(coded_msg)

encoded_message = encode_message(original_message)
encoded_message

'p enly skzyjlc ckmy enl betlle uyc xkzyc ub p lqflrelc  '

In [87]:
# function to decode message

def decode_message(msg, word_map):
  decoded_msg = []
  for ch in msg:
    decoded_ch = ch
    if ch in word_map:
      decoded_ch = word_map[ch]
    decoded_msg.append(decoded_ch)

  return ''.join(decoded_msg)

In [88]:
dna_pool = []
for _ in range(20):
  dna = list(string.ascii_lowercase)
  random.shuffle(dna)
  dna_pool.append(dna)

In [89]:
def evolve_offspring(dna_pool, n_children):
  offspring = []

  for dna in dna_pool:
    for _ in range(n_children):
      copy = dna.copy()
      j = np.random.randint(len(copy))
      k = np.random.randint(len(copy))

      tmp = copy[j]
      copy[j] = copy[k]
      copy[k] = tmp
      offspring.append(copy)

  return offspring + dna_pool



In [90]:
import numpy as np

num_iters = 1000
scores = np.zeros(num_iters)
best_dna = None
best_map = None
best_score = float('-inf')

for i in range(num_iters):
    if i > 0:
        dna_pool = evolve_offspring(dna_pool, 3)

    dna2score = {}
    for dna in dna_pool:
        current_map = {}
        for k, v in zip(letters1, dna):
            current_map[k] = v

        decoded_message = decode_message(encoded_message, current_map)
        score = get_sequence_probs(decoded_message)

        dna2score[''.join(dna)] = score

        if score > best_score:
            best_dna = dna
            best_map = current_map
            best_score = score

    scores[i] = np.mean(list(dna2score.values()))

    sorted_dna = sorted(dna2score.items(), key=lambda x: x[1], reverse=True)
    dna_pool = [list(k) for k, v in sorted_dna[:5]]

    if i % 200 == 0:
        print("iter:", i, "score:", scores[i], "best so far:", best_score)


TypeError: object of type 'numpy.float64' has no len()